### Tworzę pipeline do trenowania modelu

##### 1. Wczytuję dane z Digital Ocean Spaces

In [ ]:
# Importy do obsługi Digital Ocean Spaces
import boto3
from dotenv import load_dotenv
import os
import json
from datetime import datetime
from pathlib import Path

load_dotenv()
s3 = boto3.client("s3")

In [ ]:
# zmienne do obsługi Digital Ocean Spaces
BUCKET_NAME = "gotoitgosia13"
DATA_PREFIX = "halfmarathon/data/"
MODEL_PREFIX = "halfmarathon/models/"

In [ ]:
# pobieranie plików z Digital Ocean Spaces
response = s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix=DATA_PREFIX)
for obj in response["Contents"]:
    file_name = obj["Key"].split("/")[-1]
    print(f"Pobieram plik {file_name} z {obj['Key']}")
    s3.download_file(
        Bucket=BUCKET_NAME,
        Key=obj["Key"],
        Filename=file_name,
    )

#### 2. Czyszczenie danych

In [ ]:
# Importy do przeglądu danych i wizualizacji
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import scipy.stats as stats

In [ ]:
# Wczytanie i przegląd danych z plików CSV
df_2023 = pd.read_csv('halfmarathon_wroclaw_2023__final.csv', sep=';')
print("2023:", df_2023.shape)
df_2023.head()

In [ ]:
df_2024 = pd.read_csv('halfmarathon_wroclaw_2024__final.csv', sep=';')
print("2024:", df_2024.shape)
df_2024.head()

In [ ]:
df_2023['rok'] = 2023
df_2024['rok'] = 2024

In [ ]:
df = pd.concat([df_2023, df_2024], ignore_index=True)
print(df.shape)

In [ ]:
# Snippet pomocniczy - zmiana czasu na sekundy
def convert_time_to_seconds(time):
    if isinstance(time, (int, float)):
        return None if pd.isnull(time) else time
    if time in ['DNS', 'DNF']:
        return None
    h, m, s = time.split(':')
    return int(h) * 3600 + int(m) * 60 + int(s)

In [ ]:
df['Czas'] = df['Czas'].apply(convert_time_to_seconds)

In [ ]:
# histogram czasu na mecie z krzywą normalną
plt.figure(figsize=(10, 6))
plt.hist(df['Czas'], bins=30, density=True, alpha=0.7, edgecolor='black')

mu = df['Czas'].mean()
std = df['Czas'].std()
xmin, xmax = plt.xlim()
x = np.linspace(xmin, xmax, 100)
p = stats.norm.pdf(x, mu, std)
plt.plot(x, p, 'r', linewidth=2, label='Rozkład normalny')

plt.xlabel('Czas')
plt.ylabel('Gęstość')
plt.title('Histogram z krzywą normalną')
plt.legend()
plt.show()

In [ ]:
# macierz korelacji dla zmiennych numerycznych
correlation_matrix = df.corr(numeric_only=True)
plt.figure(figsize=(16, 12))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap='coolwarm', cbar=True)

In [ ]:
df.isna().sum()

#### 3. Dobór zmiennych i dalsze czyszczenie
> Czasy na 10/15/20 km oraz tempo i miejsca są **świadomie pominięte** – użytkownik ich nie poda, a ich użycie oznaczałoby *data leakage* (znamy je dopiero w trakcie biegu)\
> W danych skupimy się na zmiennych, na podstawie których szacowany będzie wynik. Pozostawiamy również zmienną docelową czyli czas na mecie. Wiek wyliczymy na podstawie Rocznika oraz roku w którym odbył się bieg. W związku z tym pozostawiamy następujące kolumny: Płeć, Rocznik, rok, 5 km Czas, Czas.

In [ ]:
df_model = df[['Płeć', 'Rocznik', 'rok', '5 km Czas', 'Czas']].copy()

In [ ]:
df_model['5 km Czas'] = df_model['5 km Czas'].apply(convert_time_to_seconds)
df_model['Czas'] = df_model['Czas'].apply(convert_time_to_seconds)
print(df_model.shape)
df_model.head()

In [ ]:
# oczyszczamy z brakujących danych i filtrujemy rekordy z rocznikiem 0
df_model = df_model[df_model['Rocznik'] > 1900].copy()
df_model['wiek'] = df_model['rok'] - df_model['Rocznik']
df_model = df_model.dropna()

In [ ]:
df_model.isna().sum()

In [ ]:
# zmieniamy nagłówki na język angielski i zmieniamy typ kolumny 'Płeć' na kategoryczną
df_model = df_model.rename(columns={'Płeć': 'gender', 'Rocznik': 'birth_year', 'wiek': 'age', '5 km Czas': '5km_time', 'Czas': 'finish_time', 'rok': 'halfmarathon_year'})
df_model['gender'] = df_model['gender'].astype('category')

In [ ]:
print(df_model.shape)
df_model.head()

In [ ]:
# wykres pudełkowy dla wybranych zmiennych numerycznych w df_model
df_model.boxplot()
plt.show()

In [ ]:
# oczyszczamy dane z outlierów w kolumnach '5km_time' i 'finish_time'
ratio = df_model['finish_time'] / df_model['5km_time']     # czyli ile razy cały bieg jest dłuższy od pierwszych 5 km (21,0975 / 5 = 4,22; 5km to 23,7% przy równym tempie; przeliczenie ratio przy założeniu 5km w przedziale od 15% do 30% dla całego biegu: 1 / 0,30 = 3,33 i 1 / 0,15 = 6,67); mnożnik poza [3,33; 6,67] oznacza, że pierwsze 5 km stanowi mniej niż 15% lub więcej niż 30% całego czasu.
df_model = df_model[(df_model['5km_time'] > 0) & ratio.between(3.33, 6.67)]
print(df_model.shape)

> Celowo nie zastosowałam metody IQR, gdyż IQR traktuje wszystko poza „typowym środkiem“ jako wartość odstającą. Tutaj to są jednak zwykli, prawdziwi biegacze. Aplikacja będzie miała użytkowników z czasem 3:10, a model, który takich ludzi nigdy nie widział, będzie dla nich przewidywał słabo.

#### 4. Trenowanie modelu

##### 4a. Proponowane metryki
| Metryka | Po co |
|---|---|
| **MAE** (średni błąd bezwzględny) | **główna metryka** – informuje, o ile przeciętnie model się myli w swoich przewidywaniach; łatwo ją wytłumaczyć użytkownikowi („mylimy się średnio o X min”) |
| **RMSE** (pierwiastek średniego błędu kwadratowego) | RMSE jest zawsze ≥ MAE, a jego stosunek do MAE mówi o „ogonie“ dużych pomyłek |
| **MAPE** (średni procentowy błąd bezwzględny) | wyraża błąd jako ułamek co łatwo wyrazić w procentach (np 0,04 = 4%); porównywalny dla szybkich i wolnych biegaczy |
| **R²** (współczynnik determinacji) | pokazuje, jaki ułamek zmienności czasów model wyjaśnia; 1 to ideał, a 0 to tyle, co przewidywanie średniej dla wszystkich |
| **MedAE** (mediana błędu bezwzględnego) | odporna na outliery – typowy błąd; mniej wrażliwa na skrajne pomyłki niż MAE |

Wybór modelu: **najniższe MAE w walidacji krzyżowej** (5-fold).

In [ ]:
# Importy do modelowania
import numpy as np
import pandas as pd
from pycaret.regression import setup, compare_models, create_model, plot_model, finalize_model, save_model, load_model, predict_model, pull, tune_model, add_metric, get_config
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score
from sklearn.feature_selection import mutual_info_regression
from sklearn.model_selection import RepeatedKFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance

##### 4b Kandydaci na cechy
`czas_5km_s`, `wiek`, `plec`. Sprawdzamy je trzema sposobami: korelacją / informacją wzajemną, porównaniem zestawów cech i ważnością permutacyjną.

In [ ]:
# Tak jak wspomniano wyżej do modelu trafiają tylko zmienne, które użytkownik poda, a więc płeć, wiek i czas na 5 km. Pozostawiamy również zmienną docelową czyli czas na mecie.
model_data = df_model[['gender', 'age', '5km_time', 'finish_time']].copy()

In [ ]:
# Korelacja i mutual information (spojrzenie na cechy osobno)
X_all = model_data[['5km_time', 'gender', 'age']].copy()
y = model_data["finish_time"]

mi = mutual_info_regression(
    X_all.assign(gender=(X_all["gender"] == "M").astype(int)), y, random_state=42
)
print("Korelacja z czasem na mecie:")
print(model_data.assign(gender_M=(model_data["gender"] == "M").astype(int))[["5km_time", "gender_M", "age", "finish_time"]].corr()["finish_time"].round(3))
print("\nMI", dict(zip(X_all.columns, mi.round(3))))

> Na pierwszy rzut oka widać że czas na 5 km jest zdecydowanie najważniejszy, płeć ma słaby związek, a wiek bardzo słaby.

In [ ]:
# Porównanie podzbiorów cech przy walidacji krzyżowej (co faktycznie poprawia model)
X_all = model_data[['5km_time', 'gender', 'age']].copy()
X_all['gender'] = (X_all['gender'] == 'M').astype(int)
y = model_data['finish_time']

feature_sets = {
    '5km':                    ['5km_time'],
    '5km + gender':           ['5km_time', 'gender'],
    '5km + age':              ['5km_time', 'age'],
    '5km + gender + age':     ['5km_time', 'gender', 'age'],
    'gender + age (bez 5km)': ['gender', 'age'],
}

# modele: prosty (liniowy) i nieliniowy
models = {
    'LinearRegression': LinearRegression(),
    'HistGradientBoosting': HistGradientBoostingRegressor(random_state=42),
}

# 5-krotna walidacja krzyżowa powtórzona 3 razy (15 ocen na zestaw)
cv = RepeatedKFold(n_splits=5, n_repeats=3, random_state=42)

rows = []
for fs_name, cols in feature_sets.items():
    for m_name, model in models.items():
        scores = -cross_val_score(model, X_all[cols], y, cv=cv, scoring='neg_mean_absolute_error')
        rows.append({'zestaw cech': fs_name, 'model': m_name,
                     'MAE [s]': scores.mean(), 'odch. std [s]': scores.std()})

results = pd.DataFrame(rows).round(1)
display(results.pivot(index='zestaw cech', columns='model', values='MAE [s]').loc[list(feature_sets)])
display(results)

> Cechy gender i age nie poprawiają MAE ponad sam czas na 5 km (różnice < 1 s przy rozrzucie ok. 3 s). Zostają w modelu, ponieważ aplikacja i tak zbiera te dane, a ich koszt jest zerowy.\
> Usunięcie czasu na 5 km psuje wynik prawie trzykrotnie (z ok. 300 do ok. 890 s). Ta cecha jest więc niezbędna.

In [ ]:
# Ważność permutacyjna (jak bardzo wytrenowany model polega na każdej cesze)
X_train, X_test, y_train, y_test = train_test_split(X_all, y, test_size=0.2, random_state=42)

model = HistGradientBoostingRegressor(random_state=42).fit(X_train, y_train)

from sklearn.utils import Bunch

res = permutation_importance(model, X_test, y_test,
                             scoring='neg_mean_absolute_error',
                             n_repeats=10, random_state=42)
assert isinstance(res, Bunch)

pd.DataFrame({
    'cecha': X_all.columns,
    'wzrost MAE [s]': res.importances_mean.round(1),
    'odch. std [s]': res.importances_std.round(1),
}).sort_values('wzrost MAE [s]', ascending=False)

> Czas na 5 km jest jedyną cechą, od której model realnie zależy (po jego przetasowaniu MAE rośnie o ok. 1 023 s)\
> Wiek wnosi ok. 2,5 s, a płeć nie wnosi nic mierzalnego. Potwierdza to wcześniejsze wyniki (korelacja, mutual information i porównanie podzbiorów cech).\
> Cechy gender i age pozostawiam w modelu, bo aplikacja zbiera je i tak od użytkownika, a koszt ich użycia jest zerowy.

##### 4c Eksperyment treningowy

In [ ]:
# tworzę experyment w PyCaret
exp = setup(
    data=model_data, 
    target='finish_time',
    fold=5,
    session_id=123, 
    verbose=True
)

In [ ]:
# porównujemy modele
top_models = compare_models(sort='MAE', n_select=3)

In [ ]:
for i, m in enumerate(top_models, start=1):
    print(f"{i}. miejsce: {type(m).__name__}")

In [ ]:
pull()[['Model', 'MAE', 'RMSE', 'MAPE', 'R2']].head(5)

In [ ]:
# MAE fold po foldzie dla trzech kandydatów (te same foldy)
fold_mae = {}
for model_id in ['huber', 'gbr', 'br']:
    create_model(model_id, verbose=False)
    fold_mae[model_id] = pull()['MAE'].iloc[:5].to_numpy(dtype=float)   # 5 foldów, bez Mean/Std

fold_mae = pd.DataFrame(fold_mae, index=[f'fold {i}' for i in range(5)])
fold_mae['huber lepszy od gbr o [s]'] = fold_mae['gbr'] - fold_mae['huber']
fold_mae['huber lepszy od br o [s]']  = fold_mae['br'] - fold_mae['huber']
fold_mae.round(1)

> Walidacja krzyżowa to zwykły 5-fold. Część osób biegła w obu latach i może trafić do różnych foldów. Dla modelu liniowego z 3 cechami wpływ na wynik jest pomijalny.\
\
> Huber Regressor ma najniższe MAE w walidacji krzyżowej (297.5 s ± 2.6 s) i wygrywa z Gradient Boosting oraz Bayesian Ridge w każdym z 5 foldów (średnio o ok. 3 s). Przewaga jest niewielka (~1%), ale konsekwentna. Dodatkowe zalety: prosty, szybki, odporny na nietypowych biegaczy, z czytelnymi współczynnikami.\
> Wybrany model: **Huber Regressor**.

In [ ]:
best_model = top_models[0]
best_model

In [ ]:
# Wykres błędu predykcji (error)
plot_model(best_model, plot='error')

In [ ]:
# Wykres reszt (residuals)
plot_model(best_model, plot='residuals')

In [ ]:
# Współczynniki
cols = get_config('X_train_transformed').columns
dict(zip(cols, best_model.coef_.round(3)))

> **5km_time**: **4.558**  --> każda sekunda więcej na 5 km to +4.56 s na mecie; przy kodzie 5km o 1 min wolniejszym meta jest ok. 4.5 min później\
> **age**: **2.302** --> każdy rok życia to +2.3 s przy tym samym czasie 5 km, czyli 10 lat to +23 s; starsi biegacze nieco bardziej zwalniają w drugiej części biegu\
> **gender**: **-9.359** --> przy tym samym czasie 5 km i wieku różnica między płciami to ok. 9 s, czyli praktycznie zero\
\
> To zgadza się z permutation importance: czas na 5 km robi prawie wszystko, wiek dodaje trochę, a płeć praktycznie nic.

In [ ]:
probe = pd.DataFrame({'gender': ['K', 'M'], 'age': [35, 35], '5km_time': [1500, 1500]})
predict_model(best_model, data=probe)

In [ ]:
# Wyraz wolny
best_model.intercept_

> czas na mecie = −364.5 + 4.558 × czas_5km + 2.302 × wiek − 9.359 × płeć

##### 4d Optymalizacja modelu

In [ ]:
tuned_model = tune_model(best_model, optimize='MAE', n_iter=10, choose_better=True)

In [ ]:
tuned_model

> Strojenie (10 kombinacji, random search) zmieniło epsilon z 1.35 na 1.1 i alpha z 0.0001 na 0.01. MAE spadło z 297.5 s do 297.0 s (−0.45 s), czyli znacznie mniej niż rozrzut między foldami (~3 s). Zgodnie z regułą wyboru zostaje model po strojeniu, ale praktycznie oba warianty są równoważne. Domyślne ustawienia Hubera były już bliskie optimum.

In [ ]:
# potwierdzenie na zbiorze testowym / hold-out set
predict_model(tuned_model)

> MAE na zbiorze testowym (296.0 s) jest zgodne z walidacją krzyżową (297.0 s), co potwierdza, że model dobrze uogólnia się na nowych biegaczy.

In [ ]:
# Przedział predykcji na podstawie względnych błędów modelu
X_train, y_train = get_config('X_train'), get_config('y_train')
X_test,  y_test  = get_config('X_test'),  get_config('y_test')

# 1. Stosunek: prawdziwy czas / predykcja - na zbiorze treningowym
pred_train = predict_model(tuned_model, data=X_train, verbose=False)['prediction_label'].to_numpy(dtype=float)
ratio_train = y_train.to_numpy(dtype=float) / pred_train
Q_LOW, Q_HIGH = np.quantile(ratio_train, [0.10, 0.90])
print(f"Mnożniki przedziału: dolny {Q_LOW:.3f}, górny {Q_HIGH:.3f}")

# 2. Sprawdzenie na zbiorze testowym: ilu biegaczy mieści się w przedziale? (cel: ok. 80%)
y_test_np = y_test.to_numpy(dtype=float)
pred_test = predict_model(tuned_model, data=X_test, verbose=False)['prediction_label'].to_numpy(dtype=float)
low, high = pred_test * Q_LOW, pred_test * Q_HIGH
inside = (y_test_np >= low) & (y_test_np <= high)
print(f"Pokrycie na zbiorze testowym: {inside.mean():.1%}")
print(f"Mediana szerokości przedziału: {np.median(high - low) / 60:.1f} min")

# 3. Czy przedział działa równie dobrze dla szybkich i wolnych?
speed_band = pd.qcut(X_test['5km_time'].to_numpy(dtype=float), q=5,
                     labels=['najszybsi 20%', '2', '3', '4', 'najwolniejsi 20%'])
pd.Series(inside).groupby(speed_band).mean().round(3).rename('pokrycie')

> Ogólny wynik jest dobry, ale tabela w podziale na grupy pokazuje problem.\
> Wolniejsi biegacze są proporcjonalnie mniej przewidywalni (częstsze kryzysy, marsz, nierówne tempo). Z perspektywy biznesowej to właśnie oni są główną grupą użytkowników aplikacji: amatorzy, którzy chcą wiedzieć, czego się spodziewać.\
\
> Poprawka: osobne mnożniki dla każdej grupy prędkości.

In [ ]:
# Granice 5 grup prędkości wyznaczone na zbiorze treningowym (po 20% biegaczy w każdej)
BAND_EDGES = np.quantile(X_train['5km_time'], [0.2, 0.4, 0.6, 0.8])

def speed_band_of(t5):
    # numer grupy 0-4 dla czasu na 5 km (działa też dla czasów spoza zakresu danych)
    return np.searchsorted(BAND_EDGES, t5)

# 1. Mnożniki osobno dla każdej grupy - na zbiorze treningowym
band_train = speed_band_of(X_train['5km_time'].to_numpy(dtype=float))
BAND_MULT = {}
for b in range(5):
    q_low, q_high = np.quantile(ratio_train[band_train == b], [0.10, 0.90])
    BAND_MULT[b] = (round(float(q_low), 4), round(float(q_high), 4))
display(pd.DataFrame(BAND_MULT, index=['dolny', 'górny']).T)

# 2. Sprawdzenie na zbiorze testowym
band_test = speed_band_of(X_test['5km_time'].to_numpy(dtype=float))
low2  = pred_test * np.array([BAND_MULT[b][0] for b in band_test])
high2 = pred_test * np.array([BAND_MULT[b][1] for b in band_test])
inside2 = (y_test_np >= low2) & (y_test_np <= high2)

print(f"Pokrycie ogółem: {inside2.mean():.1%}")
pd.DataFrame({
    'pokrycie': pd.Series(inside2).groupby(band_test).mean().round(3),
    'mediana szerokości [min]': pd.Series((high2 - low2) / 60).groupby(band_test).median().round(1),
}).rename(index={0: 'najszybsi 20%', 1: '2', 2: '3', 3: '4', 4: 'najwolniejsi 20%'})

> Przedział predykcji (80%) wyznaczono z empirycznych percentyli stosunku czasu rzeczywistego do przewidzianego, osobno dla 5 grup prędkości. Pokrycie na zbiorze testowym wynosi 80–82% w każdej grupie, a szerokość rośnie od ok. 10 min (najszybsi) do ok. 22 min (najwolniejsi).

#### 5. Finalizacja modelu i wysyłka do Digital Ocean Spaces

In [ ]:
# 1. Finalizacja: ponowny trening na WSZYSTKICH danych (trening + test)
final_model = finalize_model(tuned_model)
final_model

In [ ]:
# sam Huber z końca pipeline'u
huber_final = final_model.steps[-1][1]

In [ ]:
# 2. Wersja = data i godzina treningu
VERSION = datetime.now().strftime('%Y%m%d_%H%M')
model_dir = Path('models') / VERSION
model_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# 3. Zapis modelu (PyCaret dopisze rozszerzenie .pkl)
save_model(final_model, str(model_dir / 'halfmarathon_model'))

In [ ]:
# 4. Metadane
X_all = get_config('X')
cols = get_config('X_train_transformed').columns
metadata = {
    'version': VERSION,
    'model': 'HuberRegressor',
    'params': {'epsilon': float(huber_final.epsilon), 'alpha': float(huber_final.alpha)},
    'features': ['gender', 'age', '5km_time'],
    'gender_values': ['M', 'K'],
    'gender_encoding': {'M': 1, 'K': 0},
    'coefficients': dict(zip(cols, [round(float(c), 4) for c in huber_final.coef_])),
    'intercept': round(float(huber_final.intercept_), 4),
    'selection_rule': 'najniższe MAE w walidacji krzyżowej (5-fold)',
    'metrics_cv': {'MAE_s': 297.01},
    'metrics_holdout': {
        'MAE_s': round(float(mean_absolute_error(y_test, pred_test)), 2),
        'RMSE_s': round(float(mean_squared_error(y_test, pred_test) ** 0.5), 2),
        'MAPE': round(float(mean_absolute_percentage_error(y_test, pred_test)), 4),
        'R2': round(float(r2_score(y_test, pred_test)), 4),
    },
    'interval': {
        'coverage_target': 0.80,
        'coverage_holdout': round(float(inside2.mean()), 3),
        'band_edges_5km_s': [float(e) for e in BAND_EDGES],
        'band_multipliers': {str(b): list(m) for b, m in BAND_MULT.items()},
    },
    'input_ranges': {
        'age': [float(X_all['age'].min()), float(X_all['age'].max())],
        '5km_time_s': [float(X_all['5km_time'].min()), float(X_all['5km_time'].max())],
    },
    'assumption_5km': 'Model uczony na międzyczasie 5 km z półmaratonu; czas z osobnego biegu na 5 km może dawać zbyt optymistyczne predykcje.',
}
(model_dir / 'metadata.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(metadata, ensure_ascii=False, indent=2))

In [ ]:
# 5. Wysyłka do Digital Ocean Spaces + wskaźnik "latest.json"
for file in model_dir.iterdir():
    key = f"{MODEL_PREFIX}{VERSION}/{file.name}"
    print(f"Wysyłam {file} do {key}")
    s3.upload_file(
        Bucket=BUCKET_NAME,
        Key=key,
        Filename=str(file)
    )
s3.put_object(
    Bucket=BUCKET_NAME,
    Key=f"{MODEL_PREFIX}latest.json",
    Body=json.dumps({'version': VERSION}).encode('utf-8'),
    ContentType='application/json',
)
print(f"Wysłano latest.json z wersją {VERSION} do {MODEL_PREFIX}latest.json")

In [ ]:
# 6. Weryfikacja uploadu do Digital Ocean Spaces
response = s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix=MODEL_PREFIX)
remote = {obj['Key']: obj['Size'] for obj in response.get('Contents', [])}

# a) czy pliki są w Spaces
print("Pliki w Spaces:")
for key, size in remote.items():
    print(f"{key} ({size} bajtów)")

# b) porównanie z plikami lokalnymi
print("\nPorównanie z plikami lokalnymi:")
for file in model_dir.iterdir():
    key = f"{MODEL_PREFIX}{VERSION}/{file.name}"
    local_size = file.stat().st_size
    remote_size = remote.get(key)
    status = 'OK' if remote_size == local_size else 'BŁĄD: brak pliku lub inny rozmiar'
    print(f"{file.name}: lokalnie {local_size} B, w Spaces {remote_size} B -> {status}")

# c) weryfikacja czy latest.json wskazuje włąśnie zapisaną wersję
latest = json.loads(s3.get_object(
    Bucket=BUCKET_NAME, 
    Key=f"{MODEL_PREFIX}latest.json")['Body'].read()
    )
status = 'OK' if latest['version'] == VERSION else 'UWAGA: wskazuje inną wersję'
print(f"\nlatest.json -> {latest['version']} ({status})")

In [ ]:
# Test zgodności: predykcja z .pkl (PyCaret) vs równanie z metadata.json
meta = json.loads((model_dir / 'metadata.json').read_text(encoding='utf-8'))
c = meta['coefficients']

sample = get_config('X_test').sample(500, random_state=1)
pred_pkl = predict_model(final_model, data=sample, verbose=False)['prediction_label'].to_numpy(dtype=float)

gender_num = (sample['gender'].astype(str) == 'M').astype(int).to_numpy()   # założenie: M = 1, K = 0
pred_eq = (meta['intercept']
           + c['5km_time'] * sample['5km_time'].to_numpy(dtype=float)
           + c['age'] * sample['age'].to_numpy(dtype=float)
           + c['gender'] * gender_num)

max_diff = np.abs(pred_pkl - pred_eq).max()
print(f"Największa różnica: {max_diff:.3f} s")